In [36]:
import torch
import numpy as np

from dinosaw.utils import add_custom_font,  get_features
from dinosaw.wrappers import MODEL_NAMES, ModelTypes, get_models
from dinosaw.linear_probe import do_linear_probe, RampTypes, LinearProbeResult, get_ramp, gen_sample_mask

from os import listdir
from PIL import Image

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = 'cuda:0'

In [ ]:
enabled_models: tuple[ModelTypes, ...] = ('alibi_dinov3_s+_norm_wrap_ms',)

models = get_models(enabled_models, DEVICE, True, "../../models/checkpoints", "../../models/dinov3")

2026-07-24 16:00:16 | I | factory.py                 : 152 | Building wrapper 'alibi_dinov3_s+' on device cuda:0
2026-07-24 16:00:16 | I | factory.py                 : 131 | Building backbone with config: BackboneConfig(backbone_type='torch_hub', model_arch='dinov3_s+', pretrained=False, checkpoint_path='../../models/checkpoints/trained/alibi_dv3_ms.pth', model_conf_path='../../models/dinov3', stride=None, remove_pos_embed=True, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[functools.partial(<function add_alibi at 0x7f606d6d22a0>, slope_type='constant', n_reg_tokens=4, metric='euclidean', normalize=True, wrap=True, add_cls=True, jitter_mag=0.0)], dtype=torch.float32)
2026-07-24 16:00:16 | I | modifications.py           :  51 | Removed RoPE pos. embed
2026-07-24 16:00:16 | I | factory.py                 :  80 | Loading checkpoint: ../../models/checkpoints/trained/alibi_dv3_ms.pth
2026-07-24 16:00:17 | I | wrapper.py                 :  48 | Initialize

In [38]:
# ds_folder = 'data/linear_probe/homog_micros'
ds_folder = 'data/linear_probe/texture_ds'
image_files = [f for f in listdir(ds_folder)]
n_imgs = len(image_files)

replace_with_random_noise: bool = True

features = {k: [] for k in models.keys()}
for img_file in image_files:
    img_path = f'{ds_folder}/{img_file}'
    img = Image.open(img_path).convert('RGB')
    if replace_with_random_noise:
        noise_arr = np.random.randint(0, 256, (518, 518, 3), dtype=np.uint8)
        img = Image.fromarray(noise_arr).convert('RGB')

    for i, (model_name, model) in enumerate(models.items()):

        feats = get_features(model, img, channel_last=True)
        features[model_name].append(feats)

2026-07-24 16:00:17 | I | wrapper.py                 :  92 | Processing image, size: [518, 518]
2026-07-24 16:00:17 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,512,512] -> f: [1,384,32,32]
2026-07-24 16:00:17 | I | wrapper.py                 :  92 | Processing image, size: [518, 518]
2026-07-24 16:00:17 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,512,512] -> f: [1,384,32,32]
2026-07-24 16:00:17 | I | wrapper.py                 :  92 | Processing image, size: [518, 518]
2026-07-24 16:00:17 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,512,512] -> f: [1,384,32,32]
2026-07-24 16:00:17 | I | wrapper.py                 :  92 | Processing image, size: [518, 518]
2026-07-24 16:00:17 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,512,512] -> f: [1,384,32,32]
2026-07-24 16:00:17 | I | wrapper.py                 :  92 | Processing image, size: [518, 518]
2026-07-24 16:00:17 | I | wrapper.py                 : 1

In [39]:
ramp: RampTypes = 'lr+ud'
ramps_to_results: dict[ModelTypes, list[LinearProbeResult]] = {m: [] for m in enabled_models}
MASK_CUTOFF_FRAC = 1
STEP = 6
RANDOM_MASK = True
N_repeats = 6

for model_name in enabled_models:
    for i in range(n_imgs):
        feats = features[model_name][i]
        for i in range(N_repeats):
            result = do_linear_probe(feats, ramp, probe_by_channel=False, mask_step=STEP, mask_cutoff_frac=MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK, regressor='ridge')
            ramps_to_results[model_name].append(result)

In [40]:
from skimage.transform import resize
def average_results(results: list[LinearProbeResult]) -> tuple[np.ndarray, np.ndarray, float, float, np.ndarray]:
    scores_arr = np.array([res['stack_r_squared'] for res in results])
    mean_score = np.mean(scores_arr, axis=0)
    std_score = np.std(scores_arr, axis=0)

    n_pred_dims = results[0]['stack_pred'].shape[-1]
    mean_pred = np.zeros((34, 34, n_pred_dims))

    for res in results:
        pred = resize(res['stack_pred'], mean_pred.shape, order=1)
        mean_pred += pred / len(results)


    if results[0]['per_channel_scores'] is None:
        return None, None, mean_score, std_score, mean_pred

    channel_scores_arr = np.array([res['per_channel_scores'] for res in results])
    mean_channel_scores = np.mean(channel_scores_arr, axis=0)
    std_channel_scores = np.std(channel_scores_arr, axis=0)

    return mean_channel_scores, std_channel_scores, mean_score, std_score, mean_pred

In [41]:
for model_name in enabled_models:
    results = average_results(ramps_to_results[model_name])
    print(f"{model_name}: {results[2]:.2f} ± {results[3]:.2f}")

alibi_dinov3_s+: -0.86 ± 0.31


In [42]:
# Micros
# dv: 0.57 ± 0.14
# dv_b: 0.64 ± 0.12
# dv2: 0.83 ± 0.06
# dv2_b: 0.69 ± 0.10
# dv3: 0.97 ± 0.02
# vit_b: 0.71 ± 0.11
# sam_b: 0.61 ± 0.15
# vit_b_in: -0.02 ± 0.15
# deit: 0.09 ± 0.12
# clip_b: 0.11 ± 0.19
# eva02_b: 0.02 ± 0.31

# dv2: 0.83 ± 0.06
# dv2_cb: 0.78 ± 0.08
# dv2_b: 0.68 ± 0.11
# dvt: 0.75 ± 0.09
# alibi_dv2_coco: -0.23 ± 0.28

In [43]:
# Texture
# dv: 0.5310 ± 0.2101
# dv_b: 0.5915 ± 0.2079
# dv2: 0.7144 ± 0.1897
# dv2_b: 0.5770 ± 0.2212
# dv3: 0.8952 ± 0.1159
# vit_b: 0.6106 ± 0.2133
# vit_b_in: 0.0410 ± 0.2146
# sam_b: 0.6592 ± 0.2045
# deit: 0.1276 ± 0.2043
# clip_b: 0.0706 ± 0.2737
# eva02_b: -0.1760 ± 0.2764

# dv2: 0.71 ± 0.19
# dv2_cb: 0.67 ± 0.22
# dv2_b: 0.58 ± 0.23
# dvt: 0.68 ± 0.22
# alibi_dv2_coco: 0.32 ± 0.53

In [44]:
# Texutre (MLP)
# dv2: 0.40 ± 0.21
# dv2_b: 0.12 ± 0.48
# dv2_cb: 0.35 ± 0.39
# dvt: 0.36 ± 0.19
# alibi_dv2_coco: -0.85 ± 0.19

In [45]:
# Noise
# dv: 0.53 ± 0.05
# dv_b: 0.54 ± 0.05
# dv2: 0.89 ± 0.02
# dv2_b: 0.89 ± 0.03
# dv3: 0.98 ± 0.03
# vit_b: 0.71 ± 0.04
# sam_b: 0.91 ± 0.02
# vit_b_in: -0.01 ± 0.11
# deit: 0.04 ± 0.12
# clip_b: -0.04 ± 0.25
# eva02_b: -0.08 ± 0.12